In [2]:
import os, glob
import matplotlib.pyplot as plt
import scipy.io
import numpy as np
import pandas as pd

In [ ]:
dirs_datas_RA = glob.glob(os.path.join(r"./EyeMovementDetectorEvaluation/annotated_data/data used in the article/dots", "*_labelled_RA.mat"))
mat_data = scipy.io.loadmat(dirs_datas_RA[0])
print(mat_data.keys())

In [42]:
l_label = np.array([1,2,3,4,5,6])
d_label = {
    1: "Fixation",
    2: "Saccade",
    3: "Post-saccadic oscillation",
    4: "mooth pursuit",
    5: "Blink",
    6: "Undefined",
    }
sigma_threshold_scale = 0.01
pupil_threshold = 10



def calc_saccade(data_mat, sigma_threshold_scale=0.01, pupil_threshold=10, show_plot=False):
    # データ取り込み
    sampFreq = data_mat["ETdata"]["sampFreq"][0][0][0][0]
    viewDist = data_mat["ETdata"]["viewDist"][0][0][0][0]
    screenDim = data_mat["ETdata"]["screenDim"][0][0][0]
    screenRes = data_mat["ETdata"]["screenRes"][0][0][0]

    x_center, y_center = (screenRes-1)/2
    time_bin = 1/sampFreq

    df_data = pd.DataFrame(data_mat["ETdata"]["pos"][0][0])
    df_data[0] = (1/sampFreq)*df_data.index 
    df_data[5] = df_data[5].astype('int64')

    # ディスプレイの中心座標に
    df_data["pos_x"] = df_data[3] - x_center
    df_data["pos_y"] = df_data[4] - y_center

    # 角度に変換
    df_data["degree"] = calc_deg(df_data["pos_x"], df_data["pos_y"], screenRes, screenDim, viewDist)
    # 角速度に変換
    df_data["mov_deg"] = df_data["degree"].diff()/time_bin
    # 角加速度に変換
    df_data["acc_deg"] = df_data["mov_deg"].diff()/time_bin

    # 角加速度の大きさを見たいので絶対値に変換
    df_data["acc_deg_abs"] = df_data["acc_deg"].abs()

    # 5フレーム分で移動平均。移動平均は前5フレーム分
    moving_average_window = 10
    df_data["acc_deg_abs_moving_avg"] = df_data["acc_deg_abs"].rolling(moving_average_window).mean() # 前の3つとの平均なら[4:-2]

    # df_data["pupil_x_moving_avg"] = df_data[1].rolling(moving_average_window).mean()
    # df_data["pupil_y_moving_avg"] = df_data[2].rolling(moving_average_window).mean() 
    df_data["not_blink"] = (df_data[1]>pupil_threshold) | (df_data[0]>pupil_threshold)
    # display(df_data["not_blink"])

    # 閾値作成のため、標準偏差を計算
    processed_sigma = df_data["acc_deg_abs_moving_avg"][df_data["not_blink"]][2:-2].std()
    processed_mean = df_data["acc_deg_abs_moving_avg"][df_data["not_blink"]][2:-2].mean()
    # processed_mean = df_data["acc_deg_abs_moving_avg"][df_data["not_blink"]][2:-2].median()
    processed_threshold = processed_sigma*sigma_threshold_scale + processed_mean
    # 標準偏差に固定倍率をかけた値を閾値に、それ以上をサッカード。
    df_data["saccade_estimate"] = df_data["acc_deg_abs_moving_avg"][df_data["not_blink"]]>(processed_threshold)
    df_data["saccade_estimate"] = df_data["saccade_estimate"].fillna(False)
    # display(df_data["saccade_estimate"][~df_data["not_blink"]])
    # display(df_data.dtypes)
    # df_data["saccade_estimate"][df_data["acc_deg_abs_moving_avg"].isnull()] = np.nan


    
    
    if show_plot:
        for i_label in l_label:
            df_data_slice = df_data.iloc[2:-2][df_data.iloc[2:-2][5]==i_label].copy()
            plt.scatter(df_data_slice[0], df_data_slice["acc_deg_abs"], label=d_label[i_label], s=2)
        plt.plot(df_data[0].iloc[2:-2], df_data["acc_deg_abs"].iloc[2:-2], zorder=0)
        plt.plot(
            df_data[0].iloc[moving_average_window:-2], 
            df_data["acc_deg_abs_moving_avg"].iloc[moving_average_window:-2], 
            zorder=0, c="b", marker="o", markersize=2)
        
        sigma_range = 0.15
        plt.axhline(processed_threshold)
        plt.axhline(processed_mean, label="mean", c="r", zorder=0)
        # plt.ylim(0, sigma*sigma_range)
        # plt.xlim(show_range_start, show_range_end)
        
        plt.title(os.path.basename(dirs_datas_RA[0]).replace(".mat",""))
        plt.legend()
        # plt.save()
        plt.show()


    
    np_estimate = df_data["saccade_estimate"].to_numpy()
    np_estimate = np_estimate.astype(object)
    np_null = df_data["acc_deg_abs_moving_avg"].isnull().to_numpy()
    np_null = np_null.astype(object)
    np_null[-2:] = True
    np_null = np_null.astype(bool)
    np_estimate[np_null] = np.nan
    # np_null = np_null.astype(object)
    # np_null[-2:] = np.nan
    # print(np_null)

    # 回答
    np_ans = (df_data[5]==2).to_numpy()

    return np_estimate, np_ans


def calc_deg(x_mov_pix, y_mov_pix, screenRes, screenDim, viewDist):
    dist_x_mov_m = (x_mov_pix/screenRes[0]) * screenDim[0]
    dist_y_mov_m = (y_mov_pix/screenRes[1]) * screenDim[1]
    dist_mov_m = ((dist_x_mov_m**2)+(dist_y_mov_m**2))**(1/2)
    return np.degrees(np.arctan2(dist_mov_m, viewDist))


def cacl_correct(np_estimate, np_ans):
    TP = (np_estimate & np_ans).sum()
    FN = (~np_estimate & np_ans).sum()
    FP = (np_estimate & ~np_ans).sum()
    TN = (~np_estimate & ~np_ans).sum()
    # print(f"TP: {TP}, FN: {FN}, FP: {FP}, TN: {TN}")
    return TP, FN, FP, TN

def remove_nan(np_estimate, np_ans):
    np_estimate_removed = np_estimate[~pd.isna(np_estimate)].astype(bool)
    np_ans_removed      = np_ans[~pd.isna(np_estimate)].astype(bool)
    return np_estimate_removed, np_ans_removed

print("TP, FN, FP, TN")
for dir_datas_RA in dirs_datas_RA:
    data_mat = scipy.io.loadmat(dir_datas_RA)
    np_estimate, np_ans = calc_saccade(data_mat, sigma_threshold_scale=0.15, show_plot=False)
    np_estimate, np_ans = remove_nan(np_estimate, np_ans)
    TP, FN, FP, TN = cacl_correct(np_estimate, np_ans)
    print("{},{},{},{}".format(TP, FN, FP, TN))
    # break

TP, FN, FP, TN
0,78,71,1164
29,5,34,374
23,7,40,370
35,13,484,781
8,7,104,321
54,6,31,349
42,28,600,975
10,21,587,1027
65,28,368,850
8,19,52,1235
29,6,33,484
